# Task 1 — Dataset Preparation

Parse DrugBank XML (v5.1.11) to extract drugs, protein targets with gene names, indications (diseases), and gene-disease associations for network construction. Extracts the four entity types required: Drugs, Proteins (targets), Genes, and Diseases with proper relationship edges.

## 1. Initialize Project Environment

## 2. Define Configuration Parameters

In [7]:
"""Setup and imports for Lab 9 Task 1 - Dataset Preparation from DrugBank XML."""
import logging
import re
import sys
import xml.etree.ElementTree as ET
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

import pandas as pd

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

print(f"Python {sys.version}")
print(f"pandas {pd.__version__}")

Python 3.12.3 (main, Jan  8 2026, 11:30:50) [GCC 13.3.0]
pandas 2.2.3


In [8]:
@dataclass
class Task1Config:
    """Configuration for DrugBank dataset preparation."""

    handle: str = "AndreiCod"
    # DrugBank XML location (v5.1.11)
    drugbank_xml: Path = Path("../../../data/work/AndreiCod/lab08/drugbank.xml")
    export_dir: Path = Path("./artifacts")
    # Limit for development/testing (None = all drugs)
    max_drugs: Optional[int] = None  # Process all drugs from DrugBank

    def __post_init__(self):
        self.export_dir.mkdir(parents=True, exist_ok=True)

    def describe(self) -> Dict[str, str]:
        info = asdict(self)
        info["drugbank_xml"] = str(info["drugbank_xml"])
        info["export_dir"] = str(info["export_dir"])
        return info


CONFIG = Task1Config()
CONFIG.describe()

{'handle': 'AndreiCod',
 'drugbank_xml': '../../../data/work/AndreiCod/lab08/drugbank.xml',
 'export_dir': 'artifacts',
 'max_drugs': None}

## 3. Implement Core Functionality

In [9]:
# DrugBank XML namespace
NS = {"db": "http://www.drugbank.ca"}

# Known disease terms to extract from indication text (cleaned disease names)
DISEASE_KEYWORDS = {
    "cancer",
    "carcinoma",
    "tumor",
    "tumour",
    "leukemia",
    "lymphoma",
    "melanoma",
    "sarcoma",
    "diabetes",
    "hypertension",
    "asthma",
    "arthritis",
    "alzheimer",
    "parkinson",
    "depression",
    "anxiety",
    "schizophrenia",
    "epilepsy",
    "migraine",
    "infection",
    "hiv",
    "hepatitis",
    "tuberculosis",
    "malaria",
    "heart failure",
    "coronary",
    "myocardial",
    "stroke",
    "thrombosis",
    "anemia",
    "hemophilia",
    "osteoporosis",
    "psoriasis",
    "eczema",
    "crohn",
    "colitis",
    "ibd",
    "gerd",
    "ulcer",
    "multiple sclerosis",
    "lupus",
    "rheumatoid",
    "fibrosis",
    "obesity",
    "hyperlipidemia",
    "hypercholesterolemia",
    "pneumonia",
    "bronchitis",
    "copd",
    "emphysema",
    "glaucoma",
    "macular degeneration",
    "retinopathy",
    "neuropathy",
    "fibromyalgia",
    "pain",
    "inflammation",
}


def clean_disease_name(disease_text: str) -> Optional[str]:
    """
    Clean and validate disease name extraction.
    Returns cleaned disease name or None if invalid.

    Removes:
    - Reference markers like [L1234], [A5678]
    - Newlines and excessive whitespace
    - Truncated/partial entries
    - Generic terms that aren't disease names
    """
    if not disease_text or len(disease_text) < 4:
        return None

    # Clean up the text - remove newlines and normalize whitespace
    disease = disease_text.strip()
    disease = re.sub(r"[\n\r]+", " ", disease)  # Remove newlines
    disease = re.sub(r"\s+", " ", disease)  # Normalize whitespace

    # Remove DrugBank reference markers - comprehensive patterns
    # Matches: [L1234], [A5678], [F1234], [L13781, L34415], [FDA Label], etc.
    disease = re.sub(r"\s*\[[^\]]*\]", "", disease)  # Remove ALL bracketed content
    disease = re.sub(
        r"\.\s*\[.*$", "", disease
    )  # Remove truncated brackets at end (.[L...
    disease = re.sub(r"\s*\[.*$", "", disease)  # Remove any remaining unclosed brackets

    # Remove bullet points and list markers
    disease = re.sub(r"^[\-\*•:]\s*", "", disease)
    disease = re.sub(r"\s*[\-\*•:]\s*$", "", disease)

    # Remove trailing periods, commas, and parentheses artifacts
    disease = disease.rstrip(".,;:(")
    disease = re.sub(r"\s*\(\s*$", "", disease)  # Remove trailing open parenthesis

    # Clean up after removals - normalize whitespace again
    disease = re.sub(r"\s+", " ", disease)
    disease = disease.strip().lower()

    # Skip if too short after cleaning
    if len(disease) < 4:
        return None

    # Remove common prefixes that don't add value
    skip_prefixes = [
        "the treatment of",
        "treatment of",
        "treating",
        "indicated for",
        "therapy for",
        "management of",
        "prevention of",
        "prophylaxis of",
        "patients with",
        "adults with",
        "children with",
        "use in",
        "used for",
        "used in",
    ]
    for prefix in skip_prefixes:
        if disease.startswith(prefix):
            disease = disease[len(prefix) :].strip()

    # Skip generic/invalid entries
    invalid_terms = {
        "the",
        "this",
        "that",
        "adults",
        "children",
        "patients",
        "use",
        "disease",
        "condition",
        "symptoms",
        "signs",
        "unknown",
        "other",
        "contraindications to",
        "inducing",
        "locally advanced",
        "recurrent",
        "it",
        "is",
        "are",
        "was",
        "were",
        "as determined",
        "wild-type",
        "mutation-pos",
    }
    if disease in invalid_terms or len(disease) < 4:
        return None

    # Skip if disease name ends with common truncation patterns
    truncation_endings = [" as", " that", " which", " who", " in", " with", " without"]
    for ending in truncation_endings:
        if disease.endswith(ending):
            disease = disease[: -len(ending)].strip()

    # Skip entries that look like truncated sentences
    if disease.endswith((" it", " is", " are", " was", " the", " a", " an")):
        disease = disease.rsplit(" ", 1)[0]

    # Skip entries starting with articles or conjunctions (likely malformed)
    if disease.startswith(("of ", "the ", "a ", "an ", "and ", "or ", "in ")):
        return None

    # Check if it contains a known disease keyword (quality filter)
    contains_disease_term = any(kw in disease for kw in DISEASE_KEYWORDS)

    # Also accept if it looks like a proper disease name (capitalized medical terms)
    looks_like_disease = (
        len(disease) > 8 and disease[0].isalpha() and not disease.startswith("the ")
    )

    if not (contains_disease_term or looks_like_disease):
        return None

    # Truncate very long entries and clean up
    disease = disease[:60].strip()  # Reduced from 80 to 60 for cleaner names

    # Remove trailing partial words or prepositions
    trailing_words = [" of", " for", " in", " to", " with", " and", " or", " the", " a"]
    for tw in trailing_words:
        if disease.endswith(tw):
            disease = disease[: -len(tw)].strip()

    # Final validation - must have at least 5 chars and be alphabetic start
    if len(disease) < 5 or not disease[0].isalpha():
        return None

    return disease.title()


def extract_diseases_from_indication(indication_text: str) -> List[str]:
    """
    Extract disease names from indication text using improved patterns.
    Returns list of cleaned disease/condition names.
    """
    if not indication_text:
        return []

    diseases = set()
    text = indication_text.lower()

    # Pattern 1: "treatment of X" / "indicated for X"
    patterns = [
        r"(?:treatment of|indicated for|therapy (?:of|for)|management of)\s+([^.;]+?)(?:\s+(?:in patients|in adults|in children|when|and|or)|[.;,\[\]]|$)",
        r"(?:patients with|adults with|children with)\s+([^.;]+?)(?:\s+(?:who|and|or|receiving)|[.;,\[\]]|$)",
        r"(?:prevention of|prophylaxis of)\s+([^.;]+?)(?:\s+(?:in|and|or)|[.;,\[\]]|$)",
    ]

    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        for match in matches:
            cleaned = clean_disease_name(match)
            if cleaned:
                diseases.add(cleaned)

    # Pattern 2: Direct disease mentions (look for known disease terms)
    for keyword in DISEASE_KEYWORDS:
        if keyword in text:
            # Extract surrounding context
            idx = text.find(keyword)
            start = max(0, idx - 20)
            end = min(len(text), idx + len(keyword) + 30)
            context = text[start:end]

            # Find word boundaries
            words = context.split()
            for i, word in enumerate(words):
                if keyword in word.lower():
                    # Get phrase around the keyword
                    phrase_start = max(0, i - 1)
                    phrase_end = min(len(words), i + 3)
                    phrase = " ".join(words[phrase_start:phrase_end])
                    cleaned = clean_disease_name(phrase)
                    if cleaned and keyword in cleaned.lower():
                        diseases.add(cleaned)
                    break

    return list(diseases)[:7]  # Max 7 diseases per drug


def parse_drugbank_iterative(
    xml_path: Path, max_drugs: Optional[int] = None
) -> Tuple[List[Dict], List[Dict], List[Dict], List[Dict]]:
    """
    Parse DrugBank XML iteratively to extract drugs, targets, indications, and gene-disease associations.
    Uses iterparse for memory efficiency with large files.

    Returns:
        - drugs: List of drug dictionaries
        - drug_target_edges: List of drug-target relationships
        - drug_disease_edges: List of drug-disease relationships (direct indications)
        - gene_disease_edges: List of gene-disease associations (target gene linked to indicated disease)
    """
    drugs = []
    drug_target_edges = []
    drug_disease_edges = []
    gene_disease_edges = []

    # Track gene-disease pairs to avoid duplicates
    seen_gene_disease = set()

    drug_count = 0

    logging.info("Starting iterative parse of DrugBank XML: %s", xml_path)

    # Use iterparse for memory-efficient parsing
    context = ET.iterparse(str(xml_path), events=("end",))

    for event, elem in context:
        # Process drug elements
        if elem.tag == "{http://www.drugbank.ca}drug" and elem.get("type"):
            drug_id_elem = elem.find("db:drugbank-id[@primary='true']", NS)
            name_elem = elem.find("db:name", NS)
            indication_elem = elem.find("db:indication", NS)

            if drug_id_elem is None or name_elem is None:
                elem.clear()
                continue

            drug_id = drug_id_elem.text
            drug_name = name_elem.text
            indication_text = (
                indication_elem.text if indication_elem is not None else ""
            )

            # Get drug groups (approved, experimental, etc.)
            groups = [g.text for g in elem.findall("db:groups/db:group", NS)]

            drugs.append(
                {
                    "drug_id": drug_id,
                    "drug_name": drug_name,
                    "drug_type": elem.get("type"),
                    "groups": ",".join(groups),
                }
            )

            # Extract diseases from indications (cleaned)
            diseases = extract_diseases_from_indication(indication_text)
            for disease in diseases:
                drug_disease_edges.append(
                    {
                        "drug_id": drug_id,
                        "drug_name": drug_name,
                        "disease": disease,
                        "relationship": "indicated_for",
                    }
                )

            # Extract targets with gene names
            drug_genes = []  # Track genes for this drug to create gene-disease edges
            for target in elem.findall("db:targets/db:target", NS):
                target_id = target.find("db:id", NS)
                target_name = target.find("db:name", NS)
                organism = target.find("db:organism", NS)

                # Only human targets
                if organism is not None and organism.text != "Humans":
                    continue

                polypeptide = target.find("db:polypeptide", NS)
                if polypeptide is not None:
                    gene_name_elem = polypeptide.find("db:gene-name", NS)
                    uniprot_id = polypeptide.get("id", "")
                    gene_name = (
                        gene_name_elem.text if gene_name_elem is not None else ""
                    )

                    if target_name is not None and target_name.text:
                        # Get actions
                        actions = [
                            a.text for a in target.findall("db:actions/db:action", NS)
                        ]

                        drug_target_edges.append(
                            {
                                "drug_id": drug_id,
                                "drug_name": drug_name,
                                "target_id": target_id.text
                                if target_id is not None
                                else "",
                                "target_name": target_name.text,
                                "uniprot_id": uniprot_id,
                                "gene_name": gene_name,
                                "actions": ",".join(actions) if actions else "unknown",
                            }
                        )

                        # Track gene for gene-disease association
                        if gene_name:
                            drug_genes.append(
                                {
                                    "gene_name": gene_name,
                                    "target_name": target_name.text,
                                    "uniprot_id": uniprot_id,
                                }
                            )

            # Create gene-disease associations (Issue #1: Missing Gene-Disease Relationships)
            # If a drug targets a gene AND is indicated for a disease, that gene is associated with the disease
            for gene_info in drug_genes:
                for disease in diseases:
                    pair_key = (gene_info["gene_name"], disease)
                    if pair_key not in seen_gene_disease:
                        seen_gene_disease.add(pair_key)
                        gene_disease_edges.append(
                            {
                                "gene_name": gene_info["gene_name"],
                                "target_name": gene_info["target_name"],
                                "uniprot_id": gene_info["uniprot_id"],
                                "disease": disease,
                                "relationship": "associated_with",
                                "evidence": f"via drug {drug_name} ({drug_id})",
                            }
                        )

            drug_count += 1
            if drug_count % 2000 == 0:
                logging.info("Processed %d drugs...", drug_count)

            if max_drugs and drug_count >= max_drugs:
                logging.info("Reached max_drugs limit: %d", max_drugs)
                break

            # Clear element to free memory
            elem.clear()

    logging.info(
        "Parsing complete: %d drugs, %d drug-target, %d drug-disease, %d gene-disease edges",
        len(drugs),
        len(drug_target_edges),
        len(drug_disease_edges),
        len(gene_disease_edges),
    )

    return drugs, drug_target_edges, drug_disease_edges, gene_disease_edges


# Parse DrugBank XML
drugs, drug_target_edges, drug_disease_edges, gene_disease_edges = (
    parse_drugbank_iterative(CONFIG.drugbank_xml, CONFIG.max_drugs)
)

22:28:46 | INFO | Starting iterative parse of DrugBank XML: ../../../data/work/AndreiCod/lab08/drugbank.xml
22:29:14 | INFO | Processed 2000 drugs...
22:29:16 | INFO | Processed 4000 drugs...
22:29:21 | INFO | Processed 6000 drugs...
22:29:23 | INFO | Processed 8000 drugs...
22:29:33 | INFO | Processed 10000 drugs...
22:29:37 | INFO | Processed 12000 drugs...
22:29:40 | INFO | Processed 14000 drugs...
22:29:41 | INFO | Processed 16000 drugs...
22:29:41 | INFO | Parsing complete: 16575 drugs, 14802 drug-target, 5983 drug-disease, 14571 gene-disease edges


## 4. Validate with Unit Tests

In [10]:
# Convert to DataFrames
drugs_df = pd.DataFrame(drugs)
targets_df = pd.DataFrame(drug_target_edges)
diseases_df = pd.DataFrame(drug_disease_edges)
gene_diseases_df = pd.DataFrame(gene_disease_edges)

print(f"[OK] Drugs extracted: {len(drugs_df)}")
print(f"[OK] Drug-target relationships: {len(targets_df)}")
print(f"[OK] Drug-disease relationships: {len(diseases_df)}")
print(f"[OK] Gene-disease associations: {len(gene_diseases_df)}")  # NEW!
print(f"[OK] Unique targets: {targets_df['target_name'].nunique()}")
print(f"[OK] Unique genes: {targets_df['gene_name'].nunique()}")
print(f"[OK] Unique diseases: {diseases_df['disease'].nunique()}")

# Preview
print("\n=== Sample Drugs ===")
display(drugs_df.head())
print("\n=== Sample Drug-Target Relationships ===")
display(targets_df.head())
print("\n=== Sample Drug-Disease Relationships ===")
display(diseases_df.head())
print("\n=== Sample Gene-Disease Associations (NEW) ===")
display(gene_diseases_df.head())

[OK] Drugs extracted: 16575
[OK] Drug-target relationships: 14802
[OK] Drug-disease relationships: 5983
[OK] Gene-disease associations: 14571
[OK] Unique targets: 2901
[OK] Unique genes: 2794
[OK] Unique diseases: 4361

=== Sample Drugs ===


,drug_id,drug_name,drug_type,groups
0,DB00001,Lepirudin,biotech,"approved,withdrawn"
1,DB00002,Cetuximab,biotech,approved
2,DB00003,Dornase alfa,biotech,approved
3,DB00004,Denileukin diftitox,biotech,"approved,investigational"
4,DB00005,Etanercept,biotech,"approved,investigational"



=== Sample Drug-Target Relationships ===


,drug_id,drug_name,target_id,target_name,uniprot_id,gene_name,actions
0,DB00001,Lepirudin,BE0000048,Prothrombin,P00734,F2,inhibitor
1,DB00002,Cetuximab,BE0000767,Epidermal growth factor receptor,P00533,EGFR,binder
2,DB00002,Cetuximab,BE0000901,Low affinity immunoglobulin gamma Fc region re...,O75015,FCGR3B,binder
3,DB00002,Cetuximab,BE0002094,Complement C1q subcomponent subunit A,P02745,C1QA,binder
4,DB00002,Cetuximab,BE0002095,Complement C1q subcomponent subunit B,P02746,C1QB,binder



=== Sample Drug-Disease Relationships ===


,drug_id,drug_name,disease,relationship
0,DB00001,Lepirudin,Acute Coronary Syndromes (Acs) Such As Unstabl...,indicated_for
1,DB00001,Lepirudin,Anticoagulation,indicated_for
2,DB00001,Lepirudin,Heparin-Induced Thrombocytopenia (Hit),indicated_for
3,DB00001,Lepirudin,Acute Coronary Syndromes (Acs),indicated_for
4,DB00001,Lepirudin,Anticoagulation In Adult Patients With Acute C...,indicated_for



=== Sample Gene-Disease Associations (NEW) ===


,gene_name,target_name,uniprot_id,disease,relationship,evidence
0,F2,Prothrombin,P00734,Acute Coronary Syndromes (Acs) Such As Unstabl...,associated_with,via drug Lepirudin (DB00001)
1,F2,Prothrombin,P00734,Anticoagulation,associated_with,via drug Lepirudin (DB00001)
2,F2,Prothrombin,P00734,Heparin-Induced Thrombocytopenia (Hit),associated_with,via drug Lepirudin (DB00001)
3,F2,Prothrombin,P00734,Acute Coronary Syndromes (Acs),associated_with,via drug Lepirudin (DB00001)
4,F2,Prothrombin,P00734,Anticoagulation In Adult Patients With Acute C...,associated_with,via drug Lepirudin (DB00001)


In [11]:
# Validate data structure
assert len(drugs_df) > 0, "No drugs extracted"
assert len(targets_df) > 0, "No targets extracted"
assert len(gene_diseases_df) > 0, "No gene-disease associations extracted"
assert "drug_id" in drugs_df.columns, "Missing drug_id column"
assert "drug_name" in drugs_df.columns, "Missing drug_name column"
assert "target_name" in targets_df.columns, "Missing target_name column"
assert "gene_name" in targets_df.columns, "Missing gene_name column"
assert "gene_name" in gene_diseases_df.columns, "Missing gene_name in gene-disease"
assert "disease" in gene_diseases_df.columns, "Missing disease in gene-disease"

# Check for valid data
assert drugs_df["drug_id"].notna().all(), "Null drug IDs"
assert drugs_df["drug_name"].notna().all(), "Null drug names"

print("[OK] All validation checks passed")
print(
    f"[OK] Data covers drugs: {drugs_df['drug_id'].iloc[0]} to {drugs_df['drug_id'].iloc[-1]}"
)
print(
    f"[OK] Gene-disease pairs unique genes: {gene_diseases_df['gene_name'].nunique()}"
)
print(
    f"[OK] Gene-disease pairs unique diseases: {gene_diseases_df['disease'].nunique()}"
)

[OK] All validation checks passed
[OK] Data covers drugs: DB00001 to DB18711
[OK] Gene-disease pairs unique genes: 1703
[OK] Gene-disease pairs unique diseases: 3242


## 5. Export Results

In [12]:
# Export prepared data
EXPORT_DIR = CONFIG.export_dir
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Export drugs
drugs_file = EXPORT_DIR / "task1_drugs.csv"
drugs_df.to_csv(drugs_file, index=False)

# Export drug-target relationships
targets_file = EXPORT_DIR / "task1_drug_targets.csv"
targets_df.to_csv(targets_file, index=False)

# Export drug-disease relationships
diseases_file = EXPORT_DIR / "task1_drug_diseases.csv"
diseases_df.to_csv(diseases_file, index=False)

# Export gene-disease associations (NEW - Issue #1 fix)
gene_diseases_file = EXPORT_DIR / "task1_gene_diseases.csv"
gene_diseases_df.to_csv(gene_diseases_file, index=False)

print(f"[OK] Drugs saved to: {drugs_file.resolve()}")
print(f"[OK] Drug-target relationships saved to: {targets_file.resolve()}")
print(f"[OK] Drug-disease relationships saved to: {diseases_file.resolve()}")
print(f"[OK] Gene-disease associations saved to: {gene_diseases_file.resolve()}")

[OK] Drugs saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/09_repurposing/assignments/artifacts/task1_drugs.csv
[OK] Drug-target relationships saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/09_repurposing/assignments/artifacts/task1_drug_targets.csv
[OK] Drug-disease relationships saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/09_repurposing/assignments/artifacts/task1_drug_diseases.csv
[OK] Gene-disease associations saved to: /home/rbals/git/daha-bdhb/BDHB-lab/labs/09_repurposing/assignments/artifacts/task1_gene_diseases.csv
